# NexPlay — riesgo de arrepentimiento temprano al comprar un videojuego

Diplomado en Ciencia de Datos, FES Acatlán (UNAM) — Módulo V.

Este notebook corre de principio a fin en un Colab limpio: descarga dos extractos de datos publicados como assets de GitHub Releases (con verificación SHA-256) y clona el código de entrenamiento del repo a un tag fijo. No usa Google Drive, no pide credenciales y no ejecuta la ingesta de Steam — todo eso ya ocurrió para producir los extractos.

- **data-v1** (83 juegos): con él se entrena y se mide todo el modelo. Es el mismo corte con el que se entrena el modelo que sirve la API.
- **data-v2** (el catálogo ampliado): solo se usan los títulos que **no** están en data-v1, como prueba externa. No intervienen en el entrenamiento, la elección de variables, los parámetros ni los umbrales.

**Narrativa:** Problema → Datos → EDA → Calidad de datos → Ingeniería de variables → Modelo → Experimento de privacidad → Prueba externa → Conclusiones.

## 1. Problema

Cuando alguien compra un videojuego en Steam, tiene una ventana de 120 minutos de juego para pedir reembolso. NexPlay busca estimar, **antes de la compra**, el riesgo de que un jugador se arrepienta tempranamente — para eso, antes de que exista una compra real, solo puede usar dos tipos de información: lo que ya se sabe del juego (precio, descuento, recepción de la crítica) y lo que el jugador declara de sí mismo en un formulario de alta.

No observamos arrepentimiento directamente — Steam no pregunta "¿te arrepentiste?". Usamos una señal *proxy*: reseñas donde el autor jugó poco y calificó negativo.

$$Y = 1 \iff \texttt{playtime\_at\_review} < 120 \text{ min} \ \wedge\ \texttt{voted\_up} = 0$$

El umbral de 120 minutos no es arbitrario: es exactamente la ventana de reembolso de Steam. A esta señal la llamamos **arrepentimiento temprano**, nunca "abandono" ni "insatisfacción" — son cosas distintas que esta variable no puede distinguir.

## 2. Datos

### 2.1 Origen

119k+ reseñas ingeridas desde la API pública `appreviews` de Steam sobre un catálogo curado de juegos (`appids.txt` en el repo), pensado en capas:

- **Capa A** — contraste de dificultad/experiencia: juegos que la comunidad adora pero que son duros para alguien nuevo (Dark Souls, Kenshi, Dwarf Fortress) contra puertas de entrada (Stardew Valley, Hades, Portal). Sin este contraste la variable objetivo podría no tener varianza.
- **Capa B** — intersección con el corpus de Metacritic, para comparar motivos entre plataformas (fuera del alcance de este notebook).
- **Capa C** — brecha entre expectativa y recepción: lanzamientos AAA con recibimiento muy disparejo.

### 2.2 Extracto reproducible

`nexplay.db` (SQLite) no se publica: tiene texto de reseñas y vive en `datos/`, fuera de git. En su lugar, `extracto_datos.py` (en el repo) genera un extracto mínimo en Parquet — sin texto, sin `steamid`, sin nada que no haga falta para esta narrativa — y lo publicamos como *asset* de un GitHub Release con **tag fijo** (nunca `latest`, para que esta celda siga funcionando igual dentro de un año). Este notebook descarga dos de esos assets —el de **data-v1**, para entrenar y medir, y el de **data-v2**, para la prueba externa— y valida el SHA-256 de cada uno antes de tocarlo.

El extracto trae: `appid`, `nombre` (para nombrar juegos concretos en el EDA), `playtime_at_review`, `voted_up`, `timestamp_created` — reconstruyen el target y agrupan el GroupKFold — y `num_games_owned`, `es_gratis`, `precio_final`, `descuento`, `metacritic`: las columnas crudas detrás de las cinco variables del modelo de producción (conjunto `'juego'`), más `num_games_owned` cruda (no es feature del modelo, pero sin ella no se puede reproducir la bandera de privacidad de perfil ni el experimento de la sección 7).

In [ ]:
# Colab ya trae pandas, numpy, scikit-learn y pyarrow con binarios precompilados que
# coinciden entre si. Instalar versiones fijas con pip (numpy==..., scipy==..., etc.)
# rompe esa compatibilidad binaria y produce errores como
# "ImportError: cannot import name '_slice' from 'numpy._core.umath'".
# Por eso este notebook no fija ni reinstala numpy/scipy/scikit-learn: si de verdad
# falta algun paquete (no deberia, en un Colab estandar), se instala solo, sin tocarlos.
import importlib.util
import subprocess
import sys

requeridos = {"pyarrow": "pyarrow", "requests": "requests"}
faltantes = [paquete for paquete, modulo in requeridos.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

In [ ]:
# Constancia del entorno en el que corrio este notebook (Colab ya trae estos paquetes
# precompilados; no se instala nada aqui, solo se imprime lo que Colab ya tiene).
import sys

import numpy
import pandas
import pyarrow
import requests
import sklearn

print(f"python       {sys.version.split()[0]}")
print(f"numpy        {numpy.__version__}")
print(f"pandas       {pandas.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"pyarrow      {pyarrow.__version__}")
print(f"requests     {requests.__version__}")

### 2.3 Código compartido, no duplicado

El entrenamiento (`construir_features`, `construir_pipeline`, `evaluar_gkf`, `comparar_variantes_privacidad`) vive en `entrenar_baseline.py`, en el repo. Es un script normal — su `main()` está protegido por `if __name__ == "__main__":`, así que importarlo no ejecuta nada por sí solo. Este notebook clona el repo a un **tag/commit fijo** (no la rama por defecto, que puede cambiar) e importa esas mismas funciones: nunca copia la lógica.

El **código** y los **datos** tienen referencias separadas: el código se clona del tag `CODIGO_REF`; los datos de entrenamiento salen de `DATOS_ENTRENAMIENTO_REF` y los de prueba externa de `DATOS_PRUEBA_REF`, cada uno con su SHA-256. Así se puede actualizar el código sin mover los datos con los que se midió el modelo, y al revés.

In [ ]:
# --- Configuracion fija: repo publico y tags de release (NO "latest", NO una rama) ---
GITHUB_REPO = "fernandoaxelramirezgomez-coder/nexplay"

# Codigo: el tag que se clona para importar entrenar_baseline.py y entrenar_modelo.py.
CODIGO_REF = "data-v2"

# Datos de entrenamiento y metricas: data-v1, el mismo corte con el que se entrena el modelo de la API.
DATOS_ENTRENAMIENTO_REF = "data-v1"
PARQUET_ENTRENAMIENTO_SHA256 = "645d685df884c9a352e4306e899c62768ea493375d4b0378c201c7fc6d2d2d6e"

# Prueba externa: data-v2; solo se usan los titulos que no estan en data-v1.
DATOS_PRUEBA_REF = "data-v2"
PARQUET_PRUEBA_SHA256 = "025323e0d0e99031ed940e0a12218ed8e5dd9068bd4b0aedb2a805b01be2a0d1"

In [ ]:
import os

if os.path.isdir("repo_nexplay"):
    print("repo_nexplay ya existe, no se vuelve a clonar.")
else:
    !git clone --quiet --branch {CODIGO_REF} --depth 1 https://github.com/{GITHUB_REPO}.git repo_nexplay

In [ ]:
import sys

sys.path.insert(0, "repo_nexplay")

from entrenar_baseline import (  # noqa: E402 (import tras sys.path.insert, a proposito)
    N_SPLITS,
    SEMILLA,
    comparar_variantes_privacidad,
    construir_features,
    construir_pipeline,
    evaluar_gkf,
)

In [ ]:
import hashlib
from pathlib import Path

import requests


def descargar_parquet(ref: str, sha256_esperado: str) -> Path:
    """Baja nexplay_extracto.parquet del release `ref` y verifica su SHA-256 antes de usarlo."""
    url = f"https://github.com/{GITHUB_REPO}/releases/download/{ref}/nexplay_extracto.parquet"
    destino = Path(f"nexplay_extracto_{ref}.parquet")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    destino.write_bytes(resp.content)
    obtenido = hashlib.sha256(destino.read_bytes()).hexdigest()
    if obtenido != sha256_esperado:
        raise ValueError(
            f"SHA-256 de {ref} no coincide: esperado {sha256_esperado}, obtenido {obtenido}. "
            "El asset del release pudo cambiar o la descarga se corrompio; no seguir sin verificarlo."
        )
    print(f"{ref}: descarga verificada, {destino.stat().st_size / 1024:.1f} KB, sha256 OK")
    return destino


DATA_PATH = descargar_parquet(DATOS_ENTRENAMIENTO_REF, PARQUET_ENTRENAMIENTO_SHA256)
PRUEBA_PATH = descargar_parquet(DATOS_PRUEBA_REF, PARQUET_PRUEBA_SHA256)

In [ ]:
import pandas as pd

# df es data-v1: todo el EDA, la validacion y las metricas del modelo salen de aqui.
df = pd.read_parquet(DATA_PATH)
df["y"] = ((df["playtime_at_review"] < 120) & (df["voted_up"] == 0)).astype(int)

print(f"{DATOS_ENTRENAMIENTO_REF}: filas={len(df)}  juegos={df['appid'].nunique()}")
df.head()

## 3. EDA

### 3.1 Prevalencia global

In [ ]:
prevalencia_global = df["y"].mean()
print(f"prevalencia global de arrepentimiento temprano: {prevalencia_global:.4f} ({100*prevalencia_global:.2f}%)")

Clase muy desbalanceada — por eso la métrica de validación es PR-AUC, no accuracy (un modelo que siempre dice "no" acierta más del 97% de las veces sin decir nada útil).

### 3.2 Prevalencia por juego

La Capa A se armó con una hipótesis concreta: juegos duros para un jugador nuevo (Dark Souls, Kenshi, Dwarf Fortress) deberían mostrar más arrepentimiento temprano que puertas de entrada (Stardew Valley, Hades, Portal).

In [ ]:
por_juego = (
    df.groupby("nombre")["y"]
    .agg(n="size", prevalencia="mean")
    .sort_values("prevalencia")
)

duros = ["DARK SOULS™: REMASTERED", "Kenshi", "Dwarf Fortress"]
accesibles = ["Portal", "Stardew Valley", "Hades"]
por_juego.loc[duros + accesibles]

Los tres juegos elegidos por "difíciles para un novato" tienen prevalencia tan baja como los de entrada — entre 0.3% y 1.3%, todos por debajo de la media global. La dificultad del juego para alguien nuevo, sola, no separa nada.

In [ ]:
print("--- menor prevalencia ---")
display(por_juego.head(8))
print("--- mayor prevalencia ---")
display(por_juego.tail(8))

Lo que sí separa con fuerza son lanzamientos con recepción muy pareja/floja frente a la expectativa (*The Lord of the Rings: Gollum*, *WILD HEARTS*, *Redfall*, *Suicide Squad: Kill the Justice League*, *Skull and Bones*, *Battlefield 2042*), con prevalencias entre 10% y 27% — 5 a 12 veces la media global, y muy por encima de cualquier juego "difícil" de la Capa A. Es exactamente la intuición detrás de la Capa C del catálogo (brecha expectativa vs. recepción), no de la Capa A.

**Encontramos que** el eje que separa a los juegos no es qué tan duro es para un jugador nuevo, sino qué tan bien fue recibido en su lanzamiento — algo que se puede leer, en buena medida, del lado del juego (precio, descuento, nota de Metacritic) sin necesitar casi nada del perfil de quien compra. Esto reencuadra el proyecto: en vez de perfilar exhaustivamente al jugador, el modelo debe apoyarse primero en lo que transfiere del juego, y tratar el perfil declarado del jugador como contexto de afinidad, no como parte del score. Lo confirmamos con números en la sección 6 (Modelo): el conjunto sin ninguna variable de jugador retiene casi todo el PR-AUC del que sí la tiene, y por eso es el modelo de producción.

## 4. Calidad de datos

In [ ]:
priv_pct = (df["num_games_owned"] == 0).mean()
metacritic_na_pct = df["metacritic"].isna().mean()
precio_na_pct = df["precio_final"].isna().mean()

print(f"perfiles con num_games_owned == 0: {100*priv_pct:.1f}%")
print(f"juegos sin nota de metacritic: {100*metacritic_na_pct:.1f}%")
print(f"filas sin precio_final: {100*precio_na_pct:.1f}%")

**`num_games_owned == 0` es bandera de privacidad, no biblioteca vacía.** Steam no distingue "no tiene juegos" de "su biblioteca es privada" — ambos casos llegan como 0. Con ~60% de las reseñas en ese caso, imputarlo como "cero juegos" sería tratar como información algo que es, en su mayoría, ausencia de información. Por eso el proyecto lo trata con un flag explícito en vez de imputarlo en silencio (ver sección 7 — y por qué ese flag, ya evaluado, no quedó en el modelo de producción).

In [ ]:
# Juegos de pago sin precio_final (no son gratis, pero el campo llego nulo en la ingesta):
mask_precio_raro = df["precio_final"].isna() & (df["es_gratis"] == 0)
print(f"juegos de pago sin precio: {df.loc[mask_precio_raro, 'appid'].nunique()} de {df['appid'].nunique()}")
print(f"filas afectadas: {mask_precio_raro.sum():,} de {len(df):,} ({100 * mask_precio_raro.mean():.1f}%)")
df.loc[mask_precio_raro, "nombre"].unique()

Son pocos juegos (las cifras exactas, en la celda anterior) — probablemente removidos o repriceados en Steam entre la ingesta y hoy. `construir_features` los trata con `fillna(0)`, igual que a los juegos gratis, y ahí está el problema: para un juego gratuito el 0 es correcto, para uno de pago no. Es una limitación conocida; se cuantifica al final de esta sección.

**Nada de fuga temporal.** El extracto no trae `playtime_forever` ni ningún campo que solo exista porque el autor ya reseñó (`num_reviews`, `steam_purchase`, etc.) — esas columnas solo se usan en el conjunto `'completo'` de `entrenar_baseline.py`, que es apenas un chequeo interno de "¿hay señal?" y nunca llega a producción.

In [ ]:
fechas = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"reseñas entre {fechas.min().date()} y {fechas.max().date()}")

assert df["playtime_at_review"].ge(0).all(), "playtime_at_review negativo"
assert df["voted_up"].isin([0, 1]).all(), "voted_up fuera de {0,1}"
assert df["descuento"].dropna().between(0, 100).all(), "descuento fuera de [0,100]"
print("chequeos de rango: OK")

### Limitación conocida: juegos de pago con el precio imputado como 0

En los juegos de la celda anterior `appdetails` no devolvió `price_overview`, así que `precio_final` llegó nulo aunque **no son gratuitos** (`es_gratis = 0`). `construir_features` llena ese nulo con 0 y `api/scoring.py` hace lo mismo al puntuar: el modelo se entrenó con el 0 y lo sigue viendo en inferencia.

Para un juego gratuito el 0 es correcto; para uno de pago no. "De pago y a precio 0" no existe entre los juegos con precio real: queda muy por debajo de la media, el modelo lo lee como "muy barato" y le resta log-odds a esos juegos sin que sea información del juego.

**Prueba de sensibilidad** sobre el modelo de producción (conjunto `'juego'`); la celda de abajo imprime todas las cifras:

- **Imputar sin reentrenar** (mínimo, mediana y máximo de los precios de pago, solo al puntuar): cuántos juegos cambian de banda y cómo se mueve el score de los afectados.
- **Reentrenar con la mediana imputada:** PR-AUC con y sin imputar (comparado contra la desviación entre folds), cuántos juegos cambian de banda y cómo se mueven los coeficientes de precio y gratuidad.

**Decisión:** el modelo se queda con el 0 igual en entrenamiento e inferencia, y esto se documenta como limitación, mientras la celda de abajo muestre que reentrenar no mejora la métrica más allá del ruido entre folds. Lo que sí se corrigió es lo que se le dice a quien usa la UI: la ficha de esos juegos ya no muestra el precio como factor del riesgo, porque "precio por debajo del promedio" no describe al juego.

In [ ]:
import numpy as np

from entrenar_modelo import _scores_oof  # noqa: E402 (mismos cortes por terciles que usa la API)

sin_precio = df["precio_final"].isna() & (df["es_gratis"] == 0)
precios = df.loc[(df["es_gratis"] == 0) & df["precio_final"].notna()].drop_duplicates("appid")["precio_final"]
mediana_precio = precios.median()
nombres = df.drop_duplicates("appid").set_index("appid")["nombre"]

print(f"juegos de pago sin precio: {list(df.loc[sin_precio, 'nombre'].unique())}")
print(f"filas afectadas: {sin_precio.sum()} de {len(df)} ({100 * sin_precio.mean():.1f}%)")
print(f"precio_final en centavos — minimo={precios.min():.0f}  mediana={mediana_precio:.0f}  maximo={precios.max():.0f}\n")


def catalogo_neutro(X, grupos):
    """Una fila por juego: el modelo de produccion es de titulo, no hay nada del perfil que fijar."""
    return X.groupby(grupos).first()


def scores_y_bandas(modelo, juegos, cortes):
    scores = pd.Series(modelo.predict_proba(juegos)[:, 1], index=juegos.index)
    bandas = np.select([scores < cortes[0], scores < cortes[1]], ["bajo", "medio"], "alto")
    return scores, pd.Series(bandas, index=juegos.index)


# Hoy: el faltante entra como 0, en entrenamiento y en inferencia
X_hoy, y_hoy, g_hoy = construir_features(df, conjunto="juego")
cortes_hoy = np.percentile(_scores_oof(X_hoy, y_hoy, g_hoy), [100 / 3, 200 / 3])
modelo_hoy = construir_pipeline().fit(X_hoy, y_hoy)
juegos_hoy = catalogo_neutro(X_hoy, g_hoy)
scores_hoy, bandas_hoy = scores_y_bandas(modelo_hoy, juegos_hoy, cortes_hoy)
afectados = juegos_hoy.index.isin(df.loc[sin_precio, "appid"])

escalador, clf = modelo_hoy.named_steps["escalar"], modelo_hoy.named_steps["clf"]
i_precio = list(X_hoy.columns).index("log_precio_final")
z_cero = (0 - escalador.mean_[i_precio]) / escalador.scale_[i_precio]
print(f"un precio 0 queda a {z_cero:.1f} desviaciones de la media; aporte de log_precio_final al log-odds: {clf.coef_[0][i_precio] * z_cero:+.2f}\n")

# (a) Sin reentrenar: el modelo actual, con un precio imputado solo al puntuar
print("(a) sin reentrenar, precio imputado al puntuar:")
for etiqueta, precio in (("minimo", precios.min()), ("mediana", mediana_precio), ("maximo", precios.max())):
    juegos_a = juegos_hoy.copy()
    juegos_a.loc[afectados, "log_precio_final"] = np.log1p(precio)
    scores_a, bandas_a = scores_y_bandas(modelo_hoy, juegos_a, cortes_hoy)
    print(f"  precio={etiqueta:<8} juegos que cambian de banda: {(bandas_a != bandas_hoy).sum()}")
    if etiqueta == "mediana":
        for appid in juegos_hoy.index[afectados]:
            print(f"      {nombres[appid]}: score {scores_hoy[appid]:.4f} ({bandas_hoy[appid]}) -> {scores_a[appid]:.4f} ({bandas_a[appid]})")

# (b) Reentrenando con la mediana imputada en las filas afectadas
df_imp = df.copy()
df_imp.loc[sin_precio, "precio_final"] = mediana_precio
X_imp, y_imp, g_imp = construir_features(df_imp, conjunto="juego")
print("\n(b) reentrenando con la mediana imputada:")
pr_hoy = evaluar_gkf(construir_pipeline(), X_hoy, y_hoy, g_hoy, "hoy")
pr_imp = evaluar_gkf(construir_pipeline(), X_imp, y_imp, g_imp, "imputado")
cortes_imp = np.percentile(_scores_oof(X_imp, y_imp, g_imp), [100 / 3, 200 / 3])
modelo_imp = construir_pipeline().fit(X_imp, y_imp)
_, bandas_imp = scores_y_bandas(modelo_imp, catalogo_neutro(X_imp, g_imp), cortes_imp)
cambian = bandas_hoy.index[bandas_hoy != bandas_imp]

coef_hoy = pd.Series(modelo_hoy.named_steps["clf"].coef_[0], index=X_hoy.columns)
coef_imp = pd.Series(modelo_imp.named_steps["clf"].coef_[0], index=X_imp.columns)
for variable in ("log_precio_final", "es_gratis"):
    print(f"  coeficiente de {variable}: {coef_hoy[variable]:+.4f} -> {coef_imp[variable]:+.4f}")
print(f"  PR-AUC (media entre folds): hoy={pr_hoy.mean():.4f}  imputado={pr_imp.mean():.4f}  (std entre folds ~{pr_hoy.std():.3f})")
print(f"  juegos que cambian de banda: {len(cambian)} de {len(bandas_hoy)}")
for appid in cambian:
    print(f"      {nombres[appid]}: {bandas_hoy[appid]} -> {bandas_imp[appid]}")

## 5. Ingeniería de variables

`construir_features(df, conjunto="juego")` arma exactamente las cinco variables del modelo de producción: solo lo que el catálogo ya sabe del juego **antes** de la compra. Nada del jugador y nada que dependa de que la reseña ya exista. El conjunto `'compra'` agrega una sexta, `log_num_games_owned` (lo que el formulario de alta podría declarar); la sección 6 mide cuánto aporta.

In [ ]:
X, y, grupos = construir_features(df, conjunto="juego")
print("features:", list(X.columns))
X.assign(y=y).sample(5, random_state=SEMILLA)

- `log_num_games_owned` (solo en `'compra'`, no en producción): `log1p` sobre un conteo con cola larga (unos pocos perfiles declaran cientos de juegos). Si entrara al modelo, `compras_al_anio` del formulario sustituiría a `num_games_owned`; la sección 6 muestra que no aporta lo suficiente para hacerlo.
- `es_gratis`, `descuento`: ya vienen acotadas (0/1 y 0-100), sin transformar.
- `log_precio_final`: mismo `log1p`, precios van de centavos a cientos de pesos.
- `metacritic_disponible` + `metacritic`: el 27% sin nota se imputa con la mediana, pero marcado con un flag — el modelo puede aprender que "no tiene nota" es distinto de "tiene una nota mediocre", en vez de mezclarlos en silencio. El flag marca **ausencia de cobertura crítica** (Metacritic no agregó reseñas para ese título), no una propiedad del juego — no debe leerse como señal de calidad.

`grupos` es `appid`: es la clave de todo el esquema de validación de la sección 6.

## 6. Modelo

Regresión logística con `class_weight="balanced"`, sin tuning — el piso que hay que superar, no el modelo final. Validación con `GroupKFold` por `appid`: cada fold deja afuera juegos completos, así el PR-AUC mide generalización a juegos que el modelo nunca vio, no memorización de un juego particular.

In [ ]:
from sklearn.dummy import DummyClassifier

print(f"GroupKFold con N_SPLITS={N_SPLITS}, semilla={SEMILLA}\n")

print("=== conjunto 'juego' (modelo de produccion) ===")
trivial = evaluar_gkf(DummyClassifier(strategy="prior"), X, y, grupos, "juego/trivial")
logreg = evaluar_gkf(construir_pipeline(), X, y, grupos, "juego/logreg")

In [ ]:
print("=== conjunto 'compra' (lo mismo + compras declaradas, log_num_games_owned) ===")
X_compra, y_compra, grupos_compra = construir_features(df, conjunto="compra")
print("features:", list(X_compra.columns), "\n")

logreg_compra = evaluar_gkf(construir_pipeline(), X_compra, y_compra, grupos_compra, "compra/logreg")

In [ ]:
aporte_jugador = 1 - logreg.mean() / logreg_compra.mean()
print(f"juego: PR-AUC={logreg.mean():.4f}  compra: PR-AUC={logreg_compra.mean():.4f}")
print(f"quitar TODO el lado del jugador cuesta {100 * aporte_jugador:.1f}% de PR-AUC")
print(f"desviacion entre folds de 'juego': {logreg.std():.4f}  ->  diferencia de medias: {logreg_compra.mean() - logreg.mean():.4f}")

# El texto de abajo justifica la decision con estas dos cifras. Si otro corte de datos las
# cambiara, el notebook falla aqui en vez de sostener una explicacion que ya no es cierta.
assert aporte_jugador < 0.05, f"el lado del jugador ya no es marginal: {100 * aporte_jugador:.1f}% de PR-AUC"
assert abs(logreg_compra.mean() - logreg.mean()) < logreg.std(), "la diferencia ya no cabe en el ruido entre folds"

### Por qué el modelo de producción es `'juego'` (decisión B+)

La celda anterior es la justificación. Quitar **todo** el lado del jugador —`compras_al_anio`, que entra como `log_num_games_owned`— cuesta **alrededor de 2% del PR-AUC** con data-v1, y esa diferencia es **menor que la desviación entre folds**: con cinco folds agrupados por juego no se puede distinguir del ruido de qué juegos tocaron en cada fold. Es lo que sugería la sección 3: lo que separa a los juegos es cómo fueron recibidos, algo que se lee del lado del juego, no de quién lo compra.

Con eso sobre la mesa, la decisión fue quedarse con el conjunto `'juego'`:

- **Qué se gana:** un **modelo de título**, igual para cualquier persona. Su score no depende de un dato declarado que nadie verifica, se explica solo con propiedades del juego y transfiere a otras plataformas, que es justo lo que advierte `nota_plataforma` en la API ("el lado del juego transfiere").
- **Qué se pierde:** ese ~2% de PR-AUC, que no es una señal robusta.
- **Qué pasa con el perfil:** no desaparece. Sirve para contar qué tanto encaja un juego con quien lo declaró (afinidad: géneros, fricción, horas), pero **no mueve el riesgo**.

Si un corte de datos futuro hiciera que el lado del jugador aportara más que el ruido entre folds, los `assert` de la celda anterior fallan y esta decisión se tiene que revisar.

In [ ]:
from entrenar_modelo import _scores_oof  # noqa: E402 (mismos cortes por terciles que usa la API)

pipeline_final = construir_pipeline()
pipeline_final.fit(X, y)
umbral_medio, umbral_alto = np.percentile(_scores_oof(X, y, grupos), [100 / 3, 200 / 3])
print(f"umbrales de banda (tercios de scores OOF): medio={umbral_medio:.4f}  alto={umbral_alto:.4f}")
print(f"mediana de metacritic para imputar: {df['metacritic'].median()}")
coefs = pd.Series(pipeline_final.named_steps["clf"].coef_[0], index=X.columns).sort_values()
coefs

`class_weight="balanced"` reescala las clases para que el modelo aprenda con la minoría, pero eso significa que `predict_proba` **ya no es una probabilidad calibrada** — es un score útil para *ordenar* riesgo relativo, no para leerse como "38% de probabilidad de arrepentimiento". Por eso la API (`api/scoring.py`) y la UI solo exponen un nivel (BAJO/MEDIO/ALTO, calibrado por tercios de la distribución de scores de validación) y nunca un porcentaje — mostrarlo como probabilidad induciría a error.

## 7. Experimento de privacidad

`privacidad_perfil` (`num_games_owned == 0`, sección 4) se evaluó como feature explícita del conjunto `'compra'` antes de que existiera el modelo de producción. `comparar_variantes_privacidad` reproduce esa comparación ya aprobada, sin volver a experimentar: tres variantes con `GroupKFold` sobre las mismas features de `'compra'` más/menos esa bandera.

In [ ]:
resultados_privacidad = comparar_variantes_privacidad(df)

In [ ]:
for variante in ("con_privacidad", "sin_privacidad", "solo_publico"):
    r = resultados_privacidad[variante]
    print(f"{variante:<15} media={r.mean():.4f}  std={r.std():.4f}")

diferencia = resultados_privacidad["con_privacidad"].mean() - resultados_privacidad["sin_privacidad"].mean()
desviacion = resultados_privacidad["con_privacidad"].std()
print(f"\ndiferencia con/sin bandera: {diferencia:.4f}")
print(f"desviacion entre folds: {desviacion:.4f}")
print(f"tamaño de 'solo_publico' (privacidad_perfil == 0): {resultados_privacidad['n_solo_publico']} filas")
solo_publico = resultados_privacidad["solo_publico"]
print(f"'solo_publico': {100 * resultados_privacidad['n_solo_publico'] / len(df):.0f}% de las filas; "
      f"PR-AUC por fold entre {solo_publico.min():.3f} y {solo_publico.max():.3f}")
print(f"la diferencia con/sin bandera es {abs(diferencia) / desviacion:.2f} veces la desviacion entre folds")

La diferencia de PR-AUC entre incluir la bandera y no incluirla es mucho menor que la desviación entre folds (la proporción exacta, en la celda anterior): no hay señal real, es ruido de muestreo. El subconjunto `solo_publico` (perfiles no privados) además muestra folds muy inestables por tener menos positivos por fold — otra razón para no construir una regla especial alrededor de él.

**Decisión ya tomada y aplicada:** `privacidad_perfil` se sacó del conjunto `'compra'` en `entrenar_baseline.py` (se conserva solo en `'completo'`, donde se originó la comparación). Después el modelo de producción pasó al conjunto `'juego'` (sección 6), que no usa ningún dato del jugador, así que la bandera tampoco aplica ahí.

## 8. Prueba externa: los títulos de data-v2 que no están en data-v1

El catálogo creció después de entrenar: data-v2 trae títulos que el modelo nunca vio, ni siquiera dentro de un fold. Se puntúan con el modelo ya entrenado en data-v1 (`pipeline_final`, sección 6), con los umbrales de data-v1 y la mediana de Metacritic de data-v1 para imputar — exactamente lo que hace la API al servirlos. **Nada de lo que sale de aquí vuelve al modelo**: ni al entrenamiento, ni a la elección de variables, ni a los parámetros, ni a los umbrales.

Cada título se puntúa por separado para que ninguno quede fuera en silencio: se reportan los evaluables, los que tienen variables faltantes y los errores de inferencia.

In [ ]:
from sklearn.metrics import average_precision_score

df_v2 = pd.read_parquet(PRUEBA_PATH)
df_v2["y"] = ((df_v2["playtime_at_review"] < 120) & (df_v2["voted_up"] == 0)).astype(int)
nuevos = sorted(set(df_v2["appid"]) - set(df["appid"]))
assert not set(nuevos) & set(df["appid"]), "un titulo de prueba aparece en el entrenamiento"
df_prueba = df_v2[df_v2["appid"].isin(nuevos)].copy()
print(f"{DATOS_PRUEBA_REF}: {df_v2['appid'].nunique()} titulos; nuevos respecto a {DATOS_ENTRENAMIENTO_REF}: {len(nuevos)}")

mediana_entrenamiento = df["metacritic"].median()
evaluables, errores, sin_precio_de_pago, filas_por_titulo = [], [], [], {}
scores_prueba = pd.Series(index=df_prueba.index, dtype=float)
for appid, grupo in df_prueba.groupby("appid"):
    nombre = grupo["nombre"].iloc[0]
    try:
        X_t, _, _ = construir_features(grupo, conjunto="juego")
        # Se imputa con la mediana de data-v1, no con la del propio titulo: igual que la API.
        X_t["metacritic"] = grupo["metacritic"].fillna(mediana_entrenamiento)
        scores_prueba.loc[grupo.index] = pipeline_final.predict_proba(X_t[X.columns])[:, 1]
        evaluables.append(appid)
        filas_por_titulo[nombre] = len(grupo)
        if grupo["precio_final"].isna().any() and not grupo["es_gratis"].iloc[0]:
            sin_precio_de_pago.append(nombre)
    except Exception as exc:  # se reporta, no se descarta en silencio
        errores.append((nombre, repr(exc)))

y_prueba = df_prueba.loc[scores_prueba.notna(), "y"]
s_prueba = scores_prueba.dropna()
pr_auc_externo = average_precision_score(y_prueba, s_prueba)
por_titulo = s_prueba.groupby(df_prueba["appid"]).first()
bandas_prueba = np.select([por_titulo < umbral_medio, por_titulo < umbral_alto], ["bajo", "medio"], "alto")

print(f"titulos evaluables: {len(evaluables)} de {len(nuevos)}")
print(f"titulos con variables faltantes (precio de un juego de pago): {sin_precio_de_pago or 'ninguno'}")
print(f"titulos sin nota de Metacritic (es un dato, metacritic_disponible=0): {df_prueba.groupby('appid')['metacritic'].first().isna().sum()}")
print(f"errores de inferencia: {errores or 'ninguno'}")
print(f"filas: {len(y_prueba):,}  positivos: {y_prueba.sum():,}  prevalencia: {y_prueba.mean():.4f}")
print(f"PR-AUC externo: {pr_auc_externo:.4f}  (clasificador trivial = prevalencia: {y_prueba.mean():.4f}; "
      f"{pr_auc_externo / y_prueba.mean():.2f} veces)")
print(f"bandas de los titulos nuevos: {pd.Series(bandas_prueba).value_counts().to_dict()}")

La prueba externa es más dura que el GroupKFold: los títulos nuevos llegaron en otro momento y con otra mezcla de géneros y precios. Aun así el modelo ordena mejor que el clasificador trivial (celda anterior), con una ventaja más modesta que dentro de data-v1. Es la cifra honesta de cuánto transfiere el modelo a títulos que nadie usó para construirlo.

In [ ]:
print("Cifras que citan las conclusiones:")
print(f"  prevalencia en {DATOS_ENTRENAMIENTO_REF}: {prevalencia_global:.4f}")
print(f"  PR-AUC GroupKFold del modelo de produccion ('juego'): {logreg.mean():.4f} +/- {logreg.std():.4f} "
      f"({logreg.mean() / trivial.mean():.1f} veces el trivial)")
print(f"  aporte del lado del jugador ('compra' contra 'juego'): {100 * aporte_jugador:.1f}% de PR-AUC")
print(f"  variables del modelo de produccion: {len(X.columns)}")
print(f"  PR-AUC externo en {len(evaluables)} titulos nuevos: {pr_auc_externo:.4f} "
      f"({pr_auc_externo / y_prueba.mean():.2f} veces el trivial)")

## 9. Conclusiones

Las cifras exactas están en la celda anterior; aquí va lo que significan.

- **El target es una señal proxy, no arrepentimiento observado.** `playtime_at_review < 120` y `voted_up == 0` es lo más cercano que da la API de Steam a "esto no era lo que esperaba", pero no es lo mismo que preguntarle al jugador.
- **El reencuadre central de este proyecto:** el riesgo depende mucho más del juego (precio, descuento, recepción de crítica) que de quién lo compra. Sumar el lado del jugador (`'compra'`) aporta alrededor de 2% del PR-AUC, menos que la desviación entre folds — no una señal robusta (sección 6, «Por qué el modelo de producción es `'juego'`»). Por eso el modelo de producción es el conjunto `'juego'`: **el modelo estima el riesgo del título**, igual para cualquier persona. El perfil declarado en el formulario sirve para **filtrar afinidad** (qué tanto encaja el juego con lo que el jugador dice que tolera y prefiere), no para modificar el score.
- **`privacidad_perfil` se descartó con evidencia, no por intuición**: la diferencia de PR-AUC al quitarla es mucho menor que el ruido entre folds.
- **El modelo de producción (`modelo/nexplay.pkl`) es un piso, no un techo**: regresión logística sin tuning con las variables del juego, entrenada con data-v1, varias veces mejor que un clasificador trivial dentro de data-v1 y con una ventaja más modesta en la prueba externa. Con `class_weight="balanced"` el score ordena riesgo relativo pero no es una probabilidad calibrada — por eso la API expone un nivel (BAJO/MEDIO/ALTO, por tercios de la distribución de validación) y no un porcentaje.
- **Límites conocidos:** todo el entrenamiento es de reseñas de Steam (PC); no existe una fuente propia de PlayStation/Xbox/Nintendo, así que el lado del juego transfiere a otras plataformas pero el modelo no fue validado ahí (`nota_plataforma` en la API lo advierte). Tampoco se usó texto de reseña ni el corpus de Metacritic (fuente secundaria, solo para comparar motivos, nunca para entrenar).
- **Próximo paso natural:** el conjunto `'completo'` (con `num_reviews`, etc.) muestra que hay algo más de señal cuando se conocen datos posteriores a la reseña — pero esos datos no están disponibles al momento de inferencia. Vale la pena explorar features de texto o de comportamiento temprano dentro de la ventana de reembolso, no post-hoc.